<a href="https://colab.research.google.com/github/aparajitanath2/wearables_test/blob/main/ppg_motion_accuracy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
#!wget -O ppg_dalia.zip "https://archive.ics.uci.edu/static/public/495/ppg+dalia.zip"

--2026-11-07 11:45:13--  https://archive.ics.uci.edu/static/public/495/ppg+dalia.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘ppg_dalia.zip’

ppg_dalia.zip           [             <=>    ]   2.67G  41.4MB/s    in 81s     

2026-11-07 11:46:34 (33.6 MB/s) - ‘ppg_dalia.zip’ saved [2865111320]



In [18]:
#!unzip -q ppg_dalia.zip
#!unzip -q data.zip

replace data.zip? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

##### from coursera
## Statistical Inference with Confidence Intervals

Throughout week 2, we have explored the concept of confidence intervals, how to calculate them, interpret them, and what *confidence* really means.  

In this tutorial, we're going to review how to calculate confidence intervals for population proportions and means.

To begin, let's review some of the material from this week and consider once more why confidence intervals are useful tools for deriving insights from data.

First, recall the setting in which nearly all statistical analysis takes place -- the data are a sample from a population, and that population can be described in terms of various numerical parameters.  Using the data, we can estimate a parameter of interest.  For example, suppose that the parameter of interest is the mean credit card debt for all people residing in the United States.  We may call this (unknown) parameter $\theta$.  Using a sample of data, we estimate this parameter, say using the sample mean (average) of the credit card debts for all people in our sample.  We can denote this estimate by $\hat{\theta}$.  We know that $\hat{\theta}$ is not exactly equal to $\theta$, but can we somehow convey which values for $\theta$ could potentially be the actual value?  This is the goal of a confidence interval.

### Why Confidence Intervals?

A confidence interval is a calculated range around a parameter estimate (a statistic) that includes all possible true values of the parameter that are consistent with the data in a certain sense.  For example, in the lecture, we estimated, with 95% confidence, that the population proportion of parents with a toddler that use a car seat for all travel with their toddler was somewhere between 82.2% and 87.7%.  This interval is a random quantity since it is derived from one sample of data.  If we were to obtain a different sample from the same population, we would obtain a different interval.

The key property of a confidence interval is that if we were to repeatedly sample from the population, calculating a confidence interval from each sample, then 95% of our calculated confidence intervals would contain ("cover") the true proportion.

### How are Confidence Intervals Calculated?

A confidence interval for a population proportion can be calculated as follows:

$${\rm Best\ Estimate} \pm {\rm Margin\ of\ Error}$$

Where the *Best Estimate* is the **observed population proportion or mean** and the *Margin of Error* is the **t-multiplier** times the **standard error**.

The t-multiplier is calculated based on the degrees of freedom and desired confidence level.  For samples with more than 30 observations and a confidence level of 95%, we can use the z-multiplier of 1.96 instead of the t-multiplier.  The z-multiplier has the advantage of not depending on any "degrees of freedom".

The equation to create a 95% confidence interval can also be shown as:

$${\rm Population\ Proportion\ or\ Mean} \pm ({\rm t-multiplier} \cdot {\rm Standard\ Error})$$

Lastly, the Standard Error is calculated differenly for the population proportion and mean:

$${\rm Standard\ Error \ for\ Proportion} = \sqrt{\frac{{\rm Population\ Proportion} \cdot (1 - {\rm Population\ Proportion})}{{\rm Number\ Of\ Observations}}}$$

$${\rm Standard\ Error \ for\ Mean} = \frac{{\rm Standard\ Deviation}}{\sqrt{{\rm Number\ Of\ Observations}}}$$

Let's replicate the car seat example from the course lecture:

## Looking at the heart rate data
### Confidence interval for average heart rate

In [90]:
import pandas as pd
import numpy as np
import pickle as pk
from scipy import stats
import statsmodels.api as sm
import seaborn as sns

In [81]:
data = pk.load(open("PPG_FieldStudy/S2/S2.pkl", "rb"), encoding="latin1")

In [88]:
hr = data["label"]

n = len(hr)

print("Length of heart rate data: ", n)
print("True mean of heart rate for S2: ", hr.mean())

ins = [20, 100, 500, 1000]

#create a sample of 1000 measurements
#hrs = hr[:1000]
for ns in ins:
  CI_counter = 0
  for counter in range(1, 101):
    rng = np.random.default_rng(counter)
    hrs = rng.choice(data["label"], size=ns, replace=False)

    # Generating CI with 95% confidence
    tstar = stats.t.ppf(0.975, df=ns - 1)
    moe = tstar*hrs.std(ddof=1)/np.sqrt(ns)
    lcb = hrs.mean() - tstar*hrs.std(ddof=1)/np.sqrt(ns)
    ucb = hrs.mean() + tstar*hrs.std(ddof=1)/np.sqrt(ns)
    #lcb,ucb = sm.stats.DescrStatsW(hrs).tconfint_mean()
    if counter == 1:
      print("Length of sample data: ", ns)
      #print("Margin of Error: ", moe, "tstar: ", tstar)

    if hr.mean() >= lcb and hr.mean() <= ucb:
      CI_counter += 1
  print("Number of random samples collected:", counter)
  print("Number of times true mean fell within the sample CI:", CI_counter)

Length of heart rate data:  4099
True mean of heart rate for S1:  82.48588448264972
Length of sample data:  20
Number of random samples collected: 100
Number of times true mean fell within the sample CI: 96
Length of sample data:  100
Number of random samples collected: 100
Number of times true mean fell within the sample CI: 93
Length of sample data:  500
Number of random samples collected: 100
Number of times true mean fell within the sample CI: 95
Length of sample data:  1000
Number of random samples collected: 100
Number of times true mean fell within the sample CI: 99


### Proportion of heart rates that passes 100 bpm

In [98]:
#sns.displot(hr)
high = hr > 100
true_p = high.mean()
print("True proportion of HR > 100 for S2:", true_p)

ins = [20, 100, 500, 1000]

#create a sample of 1000 measurements
for ns in ins:
  CI_counter = 0
  for counter in range(1, 101):
    rng = np.random.default_rng(counter)
    sample = rng.choice(high, size=ns, replace=False)

    # Generating CI with 95% confidence
    zstar = 1.96
    moe = zstar*np.sqrt((sample.mean()*(1-sample.mean()))/ns)
    lcb = sample.mean() - moe
    ucb = sample.mean() + moe
    #lcb,ucb = sm.stats.proportion_confint(sample.sum(), ns)
    if counter == 1:
      print("Length of sample data: ", ns)
      #print("Margin of Error: ", moe, "tstar: ", tstar)

    if true_p >= lcb and true_p <= ucb:
      CI_counter += 1
  print("Number of random samples collected:", counter)
  print("Number of times true mean fell within the sample CI:", CI_counter)


True proportion of HR > 100 for S2: 0.07538424005855086
Length of sample data:  20
Number of random samples collected: 100
Number of times true mean fell within the sample CI: 77
Length of sample data:  100
Number of random samples collected: 100
Number of times true mean fell within the sample CI: 94
Length of sample data:  500
Number of random samples collected: 100
Number of times true mean fell within the sample CI: 95
Length of sample data:  1000
Number of random samples collected: 100
Number of times true mean fell within the sample CI: 99
